# Prediction Model
This code uses the DataPrep.py file to import required dataframes and then uses the elastic_net_data function to train and test the elastic net regression model on the data extracted from the original csv files. Then, it asks the user to input certain details that are required for the model to make predictions for new data.

First, the DataPrep script file is imported so all the data is available in memory. Then, other libraries and modules needed to execute the code are also imported.

In [ ]:
from DataPrep import * 

In [ ]:
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNetCV, ElasticNet

### Elastic Net Regression
The elastic net regression model was selected after performing a comparative study on several candidate models for the task of predicting conference registrations based on previous data. The elastic net model showed the best performance and was therefore selected as the model of choice for this task. Here, we create a function to perform elastic net regression on the day-wise dataframes.

In [ ]:
"""function to perform elastic net regression on the day-wise dataframes"""
def elastic_net_data (dataframe):
    max_days = dataframe["Days to Event"].max()
    # Reverse the order of the days to event column to start from 0 to enable fit_intercept = False
    dataframe["Day #"] = (max_days + 1) - dataframe["Days to Event"] 

    X = dataframe[["Day #", "Advertisement Status"]] # Create a feature matrix
    y = dataframe["Booking Count"] # Create a target vector for booking count

    # Split the data into training and testing sets. Stratify ensures that the train and test sets have similar
    # proportions of days with and without advertisements.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1, stratify = X["Advertisement Status"]) 

    # Create a new dataframe for the test set
    X_test_df = pd.DataFrame(X_test, columns = ["Day #", "Advertisement Status"])
    # Create a new series for the test set
    y_test_series = pd.Series(y_test, name = "Booking Count") 
    # Combine the testing data into a single dataframe
    test_df = pd.concat([X_test_df.reset_index(drop = True), y_test_series.reset_index(drop = True)], axis = 1)
    # Sort the testing data by days to event in ascending order
    test_df = test_df.sort_values(by = "Day #", ascending = True)  

    # Sorted feature matrix
    X_test_sorted = test_df[["Day #", "Advertisement Status"]]
    # Feature matrix for days with advertisements
    X_ad_active = X_test_sorted[X_test_sorted["Advertisement Status"] == 1] 
    # Feature matrix for days without advertisements
    X_ad_inactive = X_test_sorted[X_test_sorted["Advertisement Status"] == 0] 

    # Create an elastic net model
    regr = ElasticNetCV(l1_ratio = [.1, .3, .5, .7, .9, .95, .99], cv = 10, random_state = 1, fit_intercept = False) 

    # Fit the model to the training data
    regr.fit(X_train, y_train)
    # Make predictions for days with and without advertisements
    pred_ad_inactive = regr.predict(X_ad_inactive) 
    if X_ad_active.empty == False:
        pred_ad_active = regr.predict(X_ad_active)
    else:
        pred_ad_active = pred_ad_inactive
    
    # Get the optimal alpha value
    alpha = regr.alpha_ 
    # Get the optimal l1 ratio
    l1_ratio = regr.l1_ratio_ 
    # Calculate the effect of advertisements on the predicted booking count
    ad_effect = (pred_ad_active.mean() - pred_ad_inactive.mean()) / pred_ad_inactive.mean() 

    return X, y, l1_ratio, alpha, ad_effect


Take the user's input so that the model can make a prediction of the number of registrations and the impact of advertisements on a currently active conference campaign.  

In [ ]:
"""Create a new dataframe from user input"""
# Ask the user for the start date of the booking process
start_date = datetime.strptime(input("Enter the start date of the event in the format dd/mm/yyyy: "), "%d/%m/%Y")
# Ask the user for the current date
current_date = datetime.strptime(input("Enter the current date in the format dd/mm/yyyy: "), "%d/%m/%Y") 
# Ask the user for the event date
event_date = datetime.strptime(input("Enter the event date in the format dd/mm/yyyy: "), "%d/%m/%Y") - timedelta(days = 1) 
# Create a date range from the start date to the event date
date_range = pd.date_range(start = start_date, end = event_date)
# Create a new dataframe with the date range
input_df = pd.DataFrame(date_range, columns = ["Date"]) 
# Add a new column to the dataframe showing the days left to the event
input_df["Days to Event"] = (pd.to_datetime(event_date, dayfirst = True) - input_df["Date"]).dt.days
# Reverse the order of the days to event column to start from 1 for the model to work properly
input_df["Day #"] = (input_df["Days to Event"].max() + 1) - input_df["Days to Event"]
# Add a new column to the dataframe to indicate advertisement status. Assuming that no ads were 
# launched before current date
input_df["Advertisement Status"] = input_df["Days to Event"].apply(lambda x: 0)
# Ask the user for the booking count upto the current date
current_booking_count = int(input("Enter the booking count so far: ")) 
# Ask the user if they would like to launch an advertisement
ad_choice = input("Would you like to launch an advertisement? (yes/no): ") 
# Ask the user which audience are they targeting for the conference
audience_choice = int(input(
    "Please choose one of the following target audiences (enter the corresponding number):\n"
    "1 - IT Managers\n"
    "2 - Property Managers\n"
    "3 - Education Property Managers\n"
    "4 - Education Managers\n"
    "5 - Others\n"
    "Enter your choice (1-5): "
))

In [ ]:
# Creating a dictionary to store the audience_based dataframes
audience_data_dict = {
    1: df_ITM_days,
    2: df_PM_days,
    3: df_EPM_days,
    4: df_EM_days,
    5: df_other_days
}

# Fetching the audience dataframe based on the user input. If the user makes an invalid choice, the entire dataset
# (labelled as Others) is selected.
selected_audience_df = audience_data_dict.get(audience_choice, df_other_days)

train_x_final, train_y_final, l1_ratio, alpha, ad_effect = elastic_net_data(selected_audience_df)
# Create a new dataframe with the days from the current date to the event date
target_days_df = input_df[input_df["Date"] >= current_date].copy()
# Reset the index of the dataframe
target_days_df.reset_index(drop = True, inplace = True) 
# Create an elastic net model
regr_final = ElasticNet(l1_ratio = l1_ratio, alpha = alpha, fit_intercept = False)
# Fit the model to the training data
regr_final.fit(train_x_final, train_y_final) 
# Make predictions using the testing dataset
pred_final = regr_final.predict(target_days_df[["Day #", "Advertisement Status"]])
# Add a new column for the predicted booking count
target_days_df.loc[:, "Predicted Count"] = pred_final 
# Update the ad status to 1 for the next 7 days if the user chooses to include ads
if ad_choice.lower() == "yes":
    target_days_df.loc[target_days_df.index[:7], "Predicted Count"] *= (1 + ad_effect)  
# Calculate the total predicted bookings for the event
total_event_bookings = int(round(current_booking_count + target_days_df["Predicted Count"].sum(), 0))
# Print the total predicted bookings for the event
print("The total predicted bookings for the event are:", total_event_bookings) 